# 📰 Malawi Newspaper Article Extraction

This notebook extracts individual newspaper articles from a **single scanned PDF** and saves each article as a separate Markdown (`.md`) file.

**How it works:**
1. You give it a PDF of a scanned newspaper
2. It converts each page into an image
3. It sends each image to Claude (an AI model) and asks it to read and extract every article
4. It saves each article as its own text file, ready for search, training data, or RAG pipelines

**Before you start — install the required libraries (run this once in your terminal):**
```bash
pip install anthropic pymupdf
```

---

## Cell 1 — Configuration

This is the only cell you need to edit. It sets four variables that control how the notebook runs:

| Variable | What it does |
|---|---|
| `PDF_PATH` | The filename (or full path) of the PDF you want to process |
| `OUTPUT_FOLDER` | The folder where extracted `.md` files will be saved. It will be created automatically if it doesn't exist |
| `API_KEY` | Your Anthropic API key. You can paste it here, or leave it blank if you've set the `ANTHROPIC_API_KEY` environment variable |
| `MODEL` | Which Claude model to use. `claude-sonnet-4-6` is a good balance of speed and quality |
| `MAX_TOKENS` | The maximum length of Claude's response per page. 16,000 is enough for a full newspaper page |

> **Tip:** If you're running this in GitHub Codespaces, store your API key as a Codespaces secret (Settings → Codespaces → Secrets) so you never have to paste it into code.

In [ ]:
# ── CELL 1: Configuration ─────────────────────────────────────────────────

# Path to the PDF you want to process
PDF_PATH = "your_newspaper.pdf"   # <-- change this to your PDF filename/path

# Folder where extracted .md files will be saved
OUTPUT_FOLDER = "articles_output"

# Your Anthropic API key (or set the ANTHROPIC_API_KEY environment variable)
API_KEY = ""   # <-- paste your key here, or leave blank to use env var

# Claude model to use
MODEL = "claude-sonnet-4-6"

# Maximum tokens per Claude response
MAX_TOKENS = 16000

## Cell 2 — The Extraction Prompt

A **prompt** is the set of instructions we send to Claude along with each page image. Think of it as a very detailed job description.

This prompt does several things:

- **Tells Claude what it's looking at** — a scanned Chichewa-language newspaper
- **Defines the output format** — a JSON array (a structured list), so the code can reliably parse the response
- **Specifies exactly what fields each article must have** — headline, byline, domain, body
- **Lists what to skip** — ads, page numbers, staff credits, photo captions, section banners
- **Handles edge cases** — full-page interviews, cover pages, short news briefs, mixed ad/article pages

The instruction to return **raw JSON with no markdown fences** is important — it makes the response easy to parse programmatically. You'll see the code handle cases where Claude accidentally adds fences anyway.

In [ ]:
# ── CELL 2: The Prompt ─────────────────────────────────────────────────────

PROMPT = """This is a scanned page from a Malawi newspaper called TiKAMBE.
Articles are written in Chichewa (the local language).

Extract EVERY news article on this page — including short briefs and sidebars.
Return ONLY a JSON array — no explanation, no markdown fences, just raw JSON.

Each item must have exactly these fields:
  "headline" : the article headline as a string
  "byline"   : author name as a string, or null if not present
  "domain"   : the article's topic domain in English — choose the single best fit from:
               Politics, Health, Education, Agriculture, Sports, Crime,
               Community, Economy, Religion, Entertainment, Opinion, Obituary, Other
  "body"     : the full article body text as one string,
               with paragraph breaks as \\n\\n

════════════════════════════════════════════
WHAT TO SKIP — never include these:
════════════════════════════════════════════

  • Page numbers, dates, masthead/logo text
  • Staff credits: Mkonzi, Wopanga tsamba, Wachiwiri kwa Mkonzi, Mkonzi wa tsamba
  • Section banner labels: ZAKUKHOSI, NKHANI, MASEWERO, TIWADZIWE, ZOCHITIKA,
    ZINA UKAONA, NDEMANGA
  • The teaser row on cover pages (NKHANI / ZINA UKAONA / MASEWERO with page numbers)
  • Photo captions
  • Pull quotes
  • "Yapitilira tsamba lachiwiri" or any "continued on page N" lines
  • The Chebakali advice column box
  • Contact info, P.O. Box, email addresses, phone numbers
  • Advertisements — in ANY language (Chichewa or English)
    e.g. "Sale", "Discount", "Call now", "Mtengo", "Pagulani", "Foni"
  • Pages that are ENTIRELY advertisements — return [] immediately

════════════════════════════════════════════
SKIP THE ENTIRE PAGE — return [] if:
════════════════════════════════════════════

  • The page is entirely or mostly advertisements (Chichewa or English)
  • The page is entirely in English with no Chichewa articles
    (e.g. English-only supplements, legal notices, tenders)
  • The page contains only photos, graphics, or decorative content
  • The page is blank or unreadable

════════════════════════════════════════════
SPECIAL CASES — read carefully:
════════════════════════════════════════════

1. SPORTS/PERSONALITY INTERVIEW PAGE (Tiwadziwe / ZAKUKHOSI):
   Full-page interview = ONE article. All bold subheadings are sections
   within the interview, not separate articles.

2. COVER PAGE: Extract ALL real articles including sidebars.
   Skip teaser row, staff credits, photo captions.

3. SHORT NEWS BRIEFS ("Nkhani mwachidule"):
   Extract EACH brief as its own separate article.

4. MULTI-COLUMN ARTICLES: Reconstruct in correct reading order.

5. SIDEBAR ARTICLES: Extract separately — do not skip short ones.

6. MIXED PAGES: If a page has both ads and real articles, extract
   only the articles and skip the ads.

Return [] if the page has absolutely no extractable articles."""

## Cell 3 — Claude Client & Extraction Logic

This cell defines the functions that actually talk to Claude. There are three main pieces:

### `fix_unescaped_quotes(raw)`
A repair function. Newspaper articles often contain quotation marks (e.g. someone said "something"), which can break JSON parsing because JSON uses `"` as a special character. This function walks through the raw text character by character and escapes any quotes that appear *inside* a JSON string value rather than as JSON structure.

### `parse_json(raw, page_num)`
Tries to parse Claude's response as JSON. If that fails (which can happen with complex article text), it runs `fix_unescaped_quotes` and tries again. This two-step approach handles the most common failure mode gracefully.

### `call_claude(page_bytes, page_num)`
The main function. It:
1. **Base64-encodes** the page image — this is how you send image data as text over an API
2. **Sends it to Claude** along with the prompt from Cell 2
3. **Checks the stop reason** — if Claude stopped because it hit the token limit, we raise an error immediately (retrying won't help; you'd need a higher `MAX_TOKENS`)
4. **Retries on failure** — up to 3 times, with a short wait between attempts. JSON errors get a 5-second wait; API errors get a longer backoff (30s, 60s)
5. **Tags each article** with its source page number before returning

> **What is base64?** It's a way to convert binary data (like an image) into plain text characters. This is necessary because HTTP APIs transmit text, not raw bytes.

In [ ]:
# ── CELL 3: Claude Client & Extraction Logic ──────────────────────────────

import base64, json, os, re, time
import anthropic

# Set up the Anthropic client using the API key from Cell 1
client = anthropic.Anthropic(api_key=API_KEY if API_KEY else None)


class PageExtractionError(Exception):
    """Raised when a page cannot be extracted after all retries."""
    pass


def fix_unescaped_quotes(raw: str) -> str:
    """Escape double quotes that appear inside JSON string values."""
    result = []
    in_string = False
    escaped = False
    for i, ch in enumerate(raw):
        if escaped:
            result.append(ch)
            escaped = False
        elif ch == '\\':
            result.append(ch)
            escaped = True
        elif ch == '"':
            if in_string:
                rest = raw[i+1:].lstrip()
                if rest and rest[0] in (',', '}', ']', ':'):
                    in_string = False
                    result.append(ch)
                else:
                    result.append('\\"')
            else:
                in_string = True
                result.append(ch)
        else:
            result.append(ch)
    return ''.join(result)


def parse_json(raw: str, page_num: int) -> list:
    """Parse JSON response, with fallback quote-fixing repair."""
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"    Strict JSON failed ({e}), trying quote fix...")
        print(f"    Context around error: {repr(raw[max(0,e.pos-50):e.pos+50])}")
        fixed = fix_unescaped_quotes(raw)
        return json.loads(fixed)


def call_claude(page_bytes: bytes, page_num: int, max_retries: int = 3) -> list:
    """
    Call Claude to extract articles from a page image.
    Returns a list of article dicts. Raises PageExtractionError after all retries fail.
    """
    b64 = base64.standard_b64encode(page_bytes).decode("utf-8")

    last_error = None
    for attempt in range(1, max_retries + 1):
        print(f"    Attempt {attempt}/{max_retries}...")
        try:
            message = client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image",
                                "source": {
                                    "type": "base64",
                                    "media_type": "image/jpeg",
                                    "data": b64,
                                },
                            },
                            {"type": "text", "text": PROMPT},
                        ],
                    }
                ],
            )

            if message.stop_reason == "max_tokens":
                raise PageExtractionError(
                    f"Page {page_num}: response truncated (stop_reason='max_tokens'). "
                    f"Increase MAX_TOKENS (currently {MAX_TOKENS})."
                )

            raw = message.content[0].text
            print(f"    Raw response length: {len(raw)} characters")

            # Strip any accidental markdown fences Claude may have added
            raw = re.sub(r"^```json\s*", "", raw.strip())
            raw = re.sub(r"```$", "", raw.strip()).strip()

            articles = parse_json(raw, page_num)
            if not isinstance(articles, list):
                raise ValueError(f"Expected a JSON array, got {type(articles).__name__}")

            for a in articles:
                a["page"] = page_num
            return articles

        except (json.JSONDecodeError, ValueError) as e:
            last_error = e
            print(f"    ⚠️  Page {page_num}: parse error on attempt {attempt}: {e}")

        except anthropic.APIStatusError as e:
            last_error = e
            print(f"    ⚠️  Page {page_num}: API error on attempt {attempt}: {e}")

        except PageExtractionError:
            raise

        except Exception as e:
            last_error = e
            print(f"    ⚠️  Page {page_num}: unexpected error on attempt {attempt}: {e}")

        if attempt < max_retries:
            wait = 5 if isinstance(last_error, (json.JSONDecodeError, ValueError)) else 30 * attempt
            print(f"    Retrying in {wait}s...")
            time.sleep(wait)

    raise PageExtractionError(
        f"Page {page_num}: all {max_retries} attempts failed. Last error: {last_error}"
    )

## Cell 4 — Main Pipeline

This is the cell that actually runs the extraction. It ties everything together.

### Helper functions defined here

**`sanitize(name)`** — Strips characters that aren't allowed in filenames on any OS (e.g. `/ \ : * ? " < > |`). Article headlines become the filenames, so this prevents errors.

**`article_to_markdown(article, idx)`** — Converts one article dictionary into a nicely formatted Markdown string with a heading, byline, domain tag, page number, and body text.

### The main flow

1. **Open the PDF** using PyMuPDF (`fitz`), which is a fast Python library for reading PDF files
2. **Loop through every page** — for each page:
   - Render it as a JPEG image at 120 DPI. If the image comes out larger than 500 KB, step down to 96 DPI, then 72 DPI. Smaller images cost fewer tokens and are faster to send
   - Call `call_claude()` from Cell 3 and collect the returned articles
   - If a page fails after all retries, log it and keep going — one bad page shouldn't stop the whole run
3. **Save each article** as a numbered `.md` file in `OUTPUT_FOLDER`
   - Files are named `01_Headline.md`, `02_Headline.md`, etc.
   - The number prefix keeps them in extraction order

> **What is DPI?** Dots Per Inch — the resolution of the rendered image. Higher DPI = sharper image = larger file. 120 DPI is enough for Claude to read text clearly without sending huge files.

In [ ]:
# ── CELL 4: Main Pipeline ─────────────────────────────────────────────────

import fitz  # PyMuPDF — opens and renders PDF pages


def sanitize(name: str) -> str:
    """Remove characters unsafe for filenames."""
    return re.sub(r'[\\/*?:"<>|]', "", name).strip()


def article_to_markdown(article: dict, idx: int) -> str:
    """Convert an article dict to a Markdown string."""
    lines = []
    if article.get("headline"):
        lines.append(f"# {article['headline']}\n")
    if article.get("byline"):
        lines.append(f"**By {article['byline']}**\n")
    if article.get("domain"):
        lines.append(f"**Domain:** {article['domain']}\n")
    lines.append(f"**Page:** {article.get('page', '?')}\n")
    lines.append("---\n")
    if article.get("body"):
        lines.append(article["body"])
    return "\n".join(lines)


# ── Run ───────────────────────────────────────────────────────────────────

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}  — check PDF_PATH in Cell 1")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

doc = fitz.open(PDF_PATH)
print(f"Opened: {PDF_PATH}  ({len(doc)} pages)")
print(f"Output folder: {OUTPUT_FOLDER}\n")

all_articles = []
failed_pages = []

for page_num in range(len(doc)):
    page = doc[page_num]

    # Render the page as a JPEG — step down DPI if the image is too large
    for dpi in [120, 96, 72]:
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        pix = page.get_pixmap(matrix=mat)
        img_bytes = pix.tobytes("jpeg", jpg_quality=75)
        size_kb = len(img_bytes) / 1024
        print(f"Page {page_num + 1}/{len(doc)} — rendered at {dpi} DPI ({size_kb:.0f} KB)")
        if size_kb < 500:
            break

    print(f"Page {page_num + 1}/{len(doc)} — sending to Claude...")
    try:
        articles = call_claude(img_bytes, page_num + 1)
        all_articles.extend(articles)
        print(f"Page {page_num + 1}: {len(articles)} article(s) found\n")
    except PageExtractionError as e:
        print(f"❌ Page {page_num + 1} FAILED: {e}\n")
        failed_pages.append(page_num + 1)

doc.close()

# Save each article as a separate .md file
for idx, article in enumerate(all_articles, 1):
    headline = sanitize(article.get("headline", f"article_{idx}"))[:80]
    md_filename = f"{idx:02d}_{headline}.md"
    md_path = os.path.join(OUTPUT_FOLDER, md_filename)
    md_content = article_to_markdown(article, idx)
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(md_content)

print("─" * 50)
if failed_pages:
    print(f"⚠️  {len(failed_pages)} page(s) failed: {failed_pages}")
print(f"✅  Done — {len(all_articles)} article(s) saved to '{OUTPUT_FOLDER}/'")